# MAFFT Multiple Sequence Alignment

![MAFFT Multiple Sequence Alignment](https://proto-bio.github.io/proto-assets/images/tool/mafft/hero.png)

This notebook demonstrates `run_mafft_align`, which aligns a set of protein or nucleotide sequences by inserting gap characters to maximize positional correspondence. MAFFT provides several alignment strategies that trade off speed and accuracy, and it automatically selects the best method based on the number and length of input sequences. The resulting alignment reveals conserved positions, supports evolutionary analysis, and serves as input to homology modeling and protein engineering pipelines.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("mafft")
display_overview("mafft")
display_docs_section("mafft", "Background")

# MAFFT

[MAFFT](https://mafft.cbrc.jp/alignment/software/) (Multiple Alignment using Fast Fourier Transform) is a multiple sequence alignment program developed by Kazutaka Katoh and collaborators at Osaka University. It aligns multiple protein or nucleotide sequences by inserting gap characters so that homologous residues occupy the same alignment column, and offers a family of algorithms that trade speed for accuracy. This toolkit runs the MAFFT command-line program and returns typed MSA results.

MAFFT ([Katoh and Standley, 2013](https://doi.org/10.1093/molbev/mst010)) is a multiple sequence alignment program that constructs an alignment through progressive alignment along a guide tree followed by optional iterative refinement. Pairwise distances between input sequences are first estimated rapidly using either k-mer counting or a Fast Fourier Transform that detects homologous segments in compositionally transformed sequences. A guide tree is built from these distances, sequences are progressively aligned along the tree, and the alignment is optionally refined by an iterative cycle that repeatedly removes and re-aligns subsets of sequences.

MAFFT exposes several algorithm variants that differ in pairwise scoring and refinement strategy. `FFT-NS-i` is the default progressive method with iterative refinement on FFT-derived distances and is appropriate for large datasets. `L-INS-i` (`localpair`) performs local pairwise alignment with iterative refinement and is appropriate for sequences with one alignable domain flanked by variable regions. `G-INS-i` (`globalpair`) performs global pairwise alignment with iterative refinement and is appropriate for sequences of similar length. `E-INS-i` (`genafpair`) is a local-alignment variant that handles sequences with multiple conserved domains separated by long unalignable regions.

## Available tools

In [2]:
display_available_tools("mafft")

- **`run_mafft_align()`** — Multiple sequence alignment (MSA) using MAFFT (Multiple Alignment using Fast Fourier Transform)

### `run_mafft_align`

`run_mafft_align` takes a list of protein or nucleotide sequences and returns a multiple sequence alignment where gap characters have been inserted to maximize positional correspondence across all sequences. The `auto` method selects the most appropriate MAFFT algorithm based on input size, while `localpair`, `globalpair`, and `genafpair` provide fine-grained control over the alignment strategy. The output MSA object exposes per-column conservation scores, residue frequency tables, and methods to iterate over aligned sequences with their identifiers.

In [3]:
from proto_tools import MafftConfig, MafftInput, run_mafft_align

In [4]:
# Display input docs
display_api_reference("mafft", "input", "run_mafft_align")

# Hemoglobin subunit alpha sequences from different species
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH",  # Human
    "MVLSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHFDVSH",  # Horse
    "MVLSAADKGNVKAAWGKVGGHAAEYGAEALERMFLSFPTTKTYFPHFDLSH",  # Bovine
    "MVLSPADKTNVKAAWSKVGGHAGEYGAEALERMFLGFPTTKTYFPHFDLSH",  # Gorilla
    "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST",  # Human beta
]
sequence_ids = ["Human_alpha", "Horse_alpha", "Bovine_alpha", "Gorilla_alpha", "Human_beta"]

inputs = MafftInput(sequences=sequences, sequence_ids=sequence_ids)

**Input** — `MafftInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | List of sequences to align (minimum 2 required) |
| <code>sequence_ids</code> | <code>list[str] &#124; None</code> | <code>None</code> | Optional sequence identifiers (defaults to seq_0, seq_1, ...) |

In [5]:
# Display config docs
display_api_reference("mafft", "config", "run_mafft_align")

config = MafftConfig(align_method="auto", threads=2)

**Config** — `MafftConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>align_method</code> | <code>Literal['auto', 'localpair', 'globalpair', 'genafpair']</code> | <code>'auto'</code> | auto, localpair (L-INS-i), globalpair (G-INS-i), or genafpair (E-INS-i). |
| <code>max_iterations</code> | <code>int</code> | <code>0</code> | Iterative-refinement cycles; 0 = no refinement; ~1000 enables full *-INS-i. |
| <code>threads</code> | <code>int</code> | <code>1</code> | CPU threads for parallel processing. |
| <code>extra_args</code> | <code>list[str]</code> | <code>[]</code> | Verbatim `mafft` CLI tokens for niche flags (e.g. `['--retree', '3', '--reorder']`). |

In [6]:
# Run the alignment
result = run_mafft_align(inputs, config)

Running run_mafft_align [00:00]

In [7]:
# Display output docs
display_api_reference("mafft", "output", "run_mafft_align")

# Alignment summary
print(f"Number of sequences: {result.msa.num_sequences}")
print(f"Alignment length:    {result.msa.alignment_length}")
print(f"Total gaps:          {result.msa.total_gaps}")
print(f"Avg gap fraction:    {result.msa.average_gap_fraction:.3f}")

# Display the aligned sequences
print("\nAligned sequences:")
for seq_id, seq in result.msa.iter_with_ids():
    print(f"  {seq_id:16s} {seq}")

**Output** — `MafftOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>msa</code> | <code>MSA</code> | required | The multiple sequence alignment |

Number of sequences: 5
Alignment length:    53
Total gaps:          10
Avg gap fraction:    0.038

Aligned sequences:
  Human_alpha      MV-LSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHF-DLSH
  Horse_alpha      MV-LSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHF-DVSH
  Bovine_alpha     MV-LSAADKGNVKAAWGKVGGHAAEYGAEALERMFLSFPTTKTYFPHF-DLSH
  Gorilla_alpha    MV-LSPADKTNVKAAWSKVGGHAGEYGAEALERMFLGFPTTKTYFPHF-DLSH
  Human_beta       MVHLTPEEKSAVTALWGKV--NVDEVGGEALGRLLVVYPWTQRFFESFGDLST


## Analyze Conservation

The MSA object provides methods to analyze positional conservation across the alignment. A conservation score of 1.0 at a given column means every sequence shares the same residue at that position, indicating strong evolutionary constraint. Positions with lower conservation may correspond to regions that tolerate substitution, such as surface loops or linker regions. Identifying these patterns is a key step before designing mutations or engineering chimeric proteins.

In [8]:
msa = result.msa

# Find fully conserved positions (conservation = 1.0)
conserved_positions = []
for pos in range(msa.alignment_length):
    score = msa.get_conservation(pos)
    if score == 1.0:
        conserved_positions.append(pos)

print(f"Fully conserved positions: {len(conserved_positions)} / {msa.alignment_length}")
print(f"Conservation rate: {len(conserved_positions) / msa.alignment_length:.1%}")

# Show conservation for the first 20 positions
print("\nPer-position conservation (first 20 columns):")
print("  Position:     ", "".join(f"{i % 10}" for i in range(20)))
print("  Conservation: ", "".join(
    "*" if msa.get_conservation(i) == 1.0 else
    "." if msa.get_conservation(i) >= 0.6 else
    " " for i in range(20)
))

# Examine residue frequencies at a specific position
pos = 0
freqs = msa.get_position_frequencies(pos)
print(f"\nResidue frequencies at position {pos}: {freqs}")
column = msa.get_column(pos)
print(f"Residues at position {pos}: {column}")

Fully conserved positions: 22 / 53
Conservation rate: 41.5%

Per-position conservation (first 20 columns):
  Position:      01234567890123456789
  Conservation:  ****....* ...*.*.*.*

Residue frequencies at position 0: {'M': 1.0}
Residues at position 0: ['M', 'M', 'M', 'M', 'M']


## Export Results

The alignment can be exported to standard file formats for use in downstream tools such as phylogenetic tree builders, structure prediction pipelines, or conservation visualization software. FASTA is the most widely supported format, while A3M is a compact representation used by structure prediction tools like AlphaFold.

In [9]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export as FASTA
result.export("hemoglobin_alignment", export_path=output_dir, file_format="fasta")
print(f"Exported to {output_dir / 'hemoglobin_alignment.fasta'}")

# Preview the FASTA content
print("\nFASTA preview:")
fasta_str = result.msa.to_fasta_string()
for line in fasta_str.split("\n")[:6]:
    print(f"  {line}")

Exported to example_output/hemoglobin_alignment.fasta

FASTA preview:
  >Human_alpha
  MV-LSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHF-DLSH
  >Horse_alpha
  MV-LSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHF-DVSH
  >Bovine_alpha
  MV-LSAADKGNVKAAWGKVGGHAAEYGAEALERMFLSFPTTKTYFPHF-DLSH
